In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

import time

In [2]:
PROJECT_ROOT = (
    Path.cwd()
    .parent
)

SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [23]:
from src.data_pipeline import (
    run_data_pipeline,
)

from src.supertrend_utils import (
    generate_supertrend,
)

from src.strategy_engine import (
    run_strategy_pipeline
)

from src.cross_validation import (
    run_walk_forward_cross_validation
)

from src.plotting_utils import (
    plot_ohlcv,
    plot_ohlcv_with_supertrend_bands,
    plot_ohlcv_with_supertrend,
    plot_strategy_trades,
    plot_cumulative_pnl,
    plot_strategy_performance,
    print_performance_summary,
)

In [4]:
# 1. Load and strictly validate all OHLCV data
# (Prints descriptive statistics and timestamp reports automatically)
ohlcv_data = run_data_pipeline(
    data_dir=Path("../data"),
    print_report=True
)

DATAFRAME INFORMATION
<class 'pandas.DataFrame'>
DatetimeIndex: 1165813 entries, 2023-03-20 14:31:00 to 2026-07-03 19:59:00
Data columns (total 6 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   open         1165813 non-null  float64
 1   high         1165813 non-null  float64
 2   low          1165813 non-null  float64
 3   close        1165813 non-null  float64
 4   tick_volume  1165813 non-null  int64  
 5   source_file  1165813 non-null  str    
dtypes: float64(4), int64(1), str(1)
memory usage: 62.3 MB

DESCRIPTIVE STATISTICS
               open          high           low         close
count  1.165813e+06  1.165813e+06  1.165813e+06  1.165813e+06
mean   2.957495e+03  2.958079e+03  2.956899e+03  2.957497e+03
std    9.822508e+02  9.826735e+02  9.818091e+02  9.822510e+02
min    1.811000e+03  1.811460e+03  1.810430e+03  1.811000e+03
25%    2.032830e+03  2.033050e+03  2.032600e+03  2.032830e+03
50%    2.647580e+03  2.647960e+03  2.

In [5]:
# 2. Generate Supertrend indicator and bands
# Tunable parameters: atr_period, multiplier, smoothing_type ("RMA", "SMA", "EMA")
supertrend_data = generate_supertrend(
    ohlcv_data=ohlcv_data,
    atr_period=14,
    multiplier=3.0,
    smoothing_type="RMA"
)

# Quick check on output
supertrend_data[["close", "supertrend", "trend", "atr"]].tail()

,close,supertrend,trend,atr
time,,,,
2026-07-03 19:55:00,4175.72,4171.844521,1.0,1.141826
2026-07-03 19:56:00,4176.46,4172.564913,1.0,1.131696
2026-07-03 19:57:00,4174.02,4172.564913,1.0,1.241575
2026-07-03 19:58:00,4175.60,4172.564913,1.0,1.281462
2026-07-03 19:59:00,4174.94,4172.564913,1.0,1.262072


In [21]:
# 3. Execute the full strategy pipeline
signal_data, trades_data, metrics, plot_data = run_strategy_pipeline(
    supertrend_data=supertrend_data,
    
    # --- Shared Parameters ---
    tick_size=0.01,
    initial_capital=1000000.0,
    trading_days_per_year=252,
    
    # --- Signal Generation Parameters ---
    penetration_model="atr_fraction",             # "ticks" or "atr_fraction"
    penetration_ticks=1,
    penetration_atr_fraction=2.2,          # Used if penetration_model="atr_fraction"
    sensitivity_scalar=1.0,
    squashing_type="tanh",                 # "tanh" or "relu"
    volume_multiplier=1.5,                 # Minimum volume spike threshold
    volume_ma_period=20,
    
    # --- Backtest & Risk Management Parameters ---
    reward_risk=5.0,                       # Fixed target R:R ratio
    stop_loss_ratio=1.0,                   # Multiplier for stop distance
    position_sizing="weighted",            # "fixed" or "weighted"
    max_position_per_trade_fraction=0.10,  # Capital allocated per trade
    max_risk_per_trade_fraction=0.01,
    min_holdings_fraction=0.80,            # Liquidation threshold (80% of initial)
    transaction_costs_model="percentage",  # "percentage" or "flat"
    transaction_costs_fraction=0.0005,     # 5 bps per leg
    transaction_costs_flat=2.50,
    same_bar_priority="stop",              # "stop" or "target"
    slippage_ticks=0,
    
    # --- Performance Parameters ---
    risk_free_rate=0.02,
)

In [22]:
print_performance_summary(metrics)

          STRATEGY PERFORMANCE TEAR SHEET

[ TRADE STATISTICS ]
--------------------------------------------------
Total Trades:           49
Win Rate:               24.49%
Profit Factor:          1.23
Average R-Multiple:     -0.42R

[ ABSOLUTE RETURNS ]
--------------------------------------------------
Total Net PnL:          $2,647.79
Total Return:           0.26%
Annualized Return:      0.08%

[ RISK & RISK-ADJUSTED METRICS ]
--------------------------------------------------
Maximum Drawdown ($):   $4,951.80
Maximum Drawdown (%):   0.49%
Annualized Sharpe:      -6.66
Annualized Sortino:     -16.45
Calmar Ratio:           0.16



In [37]:
# 1. Define Window Lengths (Number of rows/bars)
# Example: 10,000 bars validation window, 2,000 bars out-of-sample test window
VALIDATION_SIZE = 100000
TEST_SIZE = 40000
INITIAL_CAPITAL = 1000000.0

# 2. Define Parameter Search Grid
# Include parameters across Supertrend, Signals, and Backtest modules
param_grid = {
    # --- Supertrend Parameters ---
    "atr_period": [10, 15, 20],
    "multiplier": [2.5, 3.0, 3.5],
    "smoothing_type": ["RMA"],
    
    # --- Signal Parameters ---
    "penetration_model": ["atr_fraction"],
    "penetration_ticks": [15],
    "penetration_atr_fraction": [1.0, 1.5, 2.0],
    "volume_multiplier": [1.5, 2.0, 5.0],
    "volume_ma_period": [20],
    
    # --- Strategy & Risk Parameters ---
    "stop_loss_ratio": [1.0],
    "reward_risk": [5.0, 10.0],  # Fixed as per your assignment constraint
    "position_sizing": ["weighted"],
    "max_position_per_trade_fraction": [0.10],
    "transaction_costs_model": ["percentage"],
    "transaction_costs_fraction": [0.0005],  # 5 bps fee
    "same_bar_priority": ["stop"],
    "slippage_ticks": [1],
    "tick_size": [0.01],
}

In [38]:
# Run Walk-Forward Optimization
(
    wfo_supertrend, 
    wfo_signals, 
    wfo_trades, 
    wfo_metrics, 
    optimal_params_log
) = run_walk_forward_cross_validation(
    ohlcv_data=ohlcv_data,
    validation_size=VALIDATION_SIZE,
    test_size=TEST_SIZE,
    param_grid=param_grid,
    target_metric="total_pnl",  # Metric to optimize on validation set
    initial_capital=INITIAL_CAPITAL,
    start_index=0
)

Executing WFO Fold 1...
Executing WFO Fold 2...
Executing WFO Fold 3...
Executing WFO Fold 4...
Executing WFO Fold 5...
Executing WFO Fold 6...
Executing WFO Fold 7...
Executing WFO Fold 8...
Executing WFO Fold 9...
Executing WFO Fold 10...
Executing WFO Fold 11...
Executing WFO Fold 12...
Executing WFO Fold 13...
Executing WFO Fold 14...
Executing WFO Fold 15...
Executing WFO Fold 16...
Executing WFO Fold 17...
Executing WFO Fold 18...
Executing WFO Fold 19...
Executing WFO Fold 20...
Executing WFO Fold 21...
Executing WFO Fold 22...
Executing WFO Fold 23...
Executing WFO Fold 24...
Executing WFO Fold 25...
Executing WFO Fold 26...

Walk-Forward Cross-Validation Complete.


In [39]:
# 1. Print Global Out-Of-Sample Performance Tear Sheet
print("\n" + "=" * 55)
print("     OUT-OF-SAMPLE WALK-FORWARD PERFORMANCE TEAR SHEET     ")
print("=" * 55)

if not wfo_metrics:
    print("⚠️ No trades were executed across the out-of-sample test windows.")
else:
    for key, value in wfo_metrics.items():
        if "percent" in key or "rate" in key:
            print(f"{key.replace('_', ' ').title():<32}: {value * 100:.2f}%")
        elif isinstance(value, float):
            print(f"{key.replace('_', ' ').title():<32}: {value:.2f}")
        else:
            print(f"{key.replace('_', ' ').title():<32}: {value}")

# 2. Display Optimal Parameters Per Fold
print("\n" + "=" * 55)
print("          OPTIMAL PARAMETERS SELECTED PER FOLD          ")
print("=" * 55)

params_df = pd.DataFrame([
    {
        "Fold": item["fold"],
        "Test Start": item["start_time"],
        "Test End": item["end_time"],
        **item["best_params"]
    }
    for item in optimal_params_log
])

display(params_df)


     OUT-OF-SAMPLE WALK-FORWARD PERFORMANCE TEAR SHEET     
Total Trades                    : 72
Win Rate                        : 12.50%
Profit Factor                   : 0.72
Average R Multiple              : -0.86
Total Pnl                       : -5108.05
Total Return Percent            : -0.51%
Annualized Return Percent       : -0.17%
Maximum Drawdown Absolute       : 9742.65
Maximum Drawdown Percent        : 0.97%
Annualized Sharpe Ratio         : -0.47
Annualized Sortino Ratio        : -0.51
Calmar Ratio                    : -0.18

          OPTIMAL PARAMETERS SELECTED PER FOLD          


,Fold,Test Start,Test End,atr_period,multiplier,smoothing_type,penetration_model,penetration_ticks,penetration_atr_fraction,volume_multiplier,volume_ma_period,stop_loss_ratio,reward_risk,position_sizing,max_position_per_trade_fraction,transaction_costs_model,transaction_costs_fraction,same_bar_priority,slippage_ticks,tick_size
0,1,2023-06-30 12:06:00,2023-08-10 15:54:00,10,3.5,RMA,atr_fraction,15,2.0,2.0,20,1.0,5.0,weighted,0.1,percentage,0.0005,stop,1,0.01
1,2,2023-08-10 15:55:00,2023-09-20 20:44:00,15,2.5,RMA,atr_fraction,15,1.5,5.0,20,1.0,5.0,weighted,0.1,percentage,0.0005,stop,1,0.01
2,3,2023-09-20 20:45:00,2023-10-31 21:21:00,15,2.5,RMA,atr_fraction,15,1.5,5.0,20,1.0,5.0,weighted,0.1,percentage,0.0005,stop,1,0.01
3,4,2023-10-31 21:22:00,2023-12-12 04:43:00,15,2.5,RMA,atr_fraction,15,2.0,2.0,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
4,5,2023-12-12 04:44:00,2024-01-24 07:33:00,10,3.0,RMA,atr_fraction,15,1.5,2.0,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
5,6,2024-01-24 07:34:00,2024-03-05 10:35:00,15,2.5,RMA,atr_fraction,15,2.0,1.5,20,1.0,5.0,weighted,0.1,percentage,0.0005,stop,1,0.01
6,7,2024-03-05 10:36:00,2024-04-16 11:00:00,15,2.5,RMA,atr_fraction,15,2.0,1.5,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
7,8,2024-04-16 11:01:00,2024-05-27 11:23:00,20,2.5,RMA,atr_fraction,15,2.0,2.0,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
8,9,2024-05-27 11:24:00,2024-07-05 19:13:00,20,2.5,RMA,atr_fraction,15,2.0,2.0,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
9,10,2024-07-05 19:14:00,2024-08-15 19:35:00,15,2.5,RMA,atr_fraction,15,1.0,5.0,20,1.0,10.0,weighted,0.1,percentage,0.0005,stop,1,0.01
